In [33]:
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

import numpy as np
from tslearn.clustering import TimeSeriesKMeans
from tslearn.datasets import CachedDatasets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance, \
    TimeSeriesResampler
import seaborn as sns
from tslearn.utils import to_time_series_dataset
from tslearn.clustering import silhouette_score

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score 
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import sklearn.cluster

import os
from yellowbrick.cluster import SilhouetteVisualizer

import math
import scipy

from sklearn.metrics import adjusted_rand_score

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from tslearn.metrics import soft_dtw

In [34]:
def ts_cluster_visualization(y_pred, df, n_clusters, plot_title):
    ts_size = df.shape[1]
    ts_max = df.max()
    plt.figure()
    for cluster in range(n_clusters):
        plt.subplot(4, math.ceil(n_clusters/4), cluster+1)
        for ts in df[y_pred == cluster]:
            plt.plot(ts.ravel(), "k-", alpha=.2)
        plt.plot(np.mean(df[y_pred == cluster], axis=0), "r-")
        plt.xlim(0, ts_size)
        plt.ylim(0, ts_max)
        plt.text(0.55, 0.35,'Cluster %d' % (cluster),
                 transform=plt.gca().transAxes)
        if cluster == 1:
            plt.title(plot_title)      
    plt.tight_layout()
    plt.show()

In [35]:
def ts_cluster_distance(y_pred, df, n_clusters, distance_measure, gamma=1.0):
    distances = list()
    for cluster in range(n_clusters):
        cluster_center = np.mean(df[y_pred == cluster], axis=0)
        for ts in df[y_pred == cluster]:
            if(distance_measure == "Euclidean"):
                diff = np.sqrt(np.sum((ts.ravel() - cluster_center)**2))
            elif distance_measure == "SoftDTW":
                diff = soft_dtw(ts.reshape(-1, 1), cluster_center.reshape(-1, 1), gamma=gamma)
            else:
                raise ValueError("Unsupported distance measure: choose 'Euclidean' or 'SoftDTW'")
            distances.append(diff)
    return np.mean(distances)

In [36]:
def import_ff_data(filename):
    expected_columns=155
    data = []
    with open(filename, 'r') as file:
        for line in file:
            row = line.strip().split(',')
            if len(row) < expected_columns:
                row += [np.nan] * (expected_columns - len(row))
            data.append(row)
    df = pd.DataFrame(data)
    def fill_last_valid(row):
        for i in range(1, len(row)):
            if pd.isna(row[i]):
                row[i] = row[i - 1]  
        return row
    df_filled = df.apply(fill_last_valid, axis=1)
    return df_filled

In [37]:
def ts_cluster_distance(y_pred, df1, df2, df3, n_clusters, distance_measure, gamma=1.0):
    distances = []

    for cluster in range(n_clusters):
        mask = (y_pred == cluster)
        if not np.any(mask):
            continue  # skip empty clusters

        # cluster centers
        c1 = np.mean(df1[mask], axis=0)
        c2 = np.mean(df2[mask], axis=0)
        c3 = np.mean(df3[mask], axis=0)

        if distance_measure == "Euclidean":
            for ts1, ts2, ts3 in zip(df1[mask], df2[mask], df3[mask]):
                diff = np.sqrt(
                    np.sum((ts1.ravel() - c1) ** 2) +
                    np.sum((ts2.ravel() - c2) ** 2) +
                    np.sum((ts3.ravel() - c3) ** 2)
                )
                distances.append(diff)

        elif distance_measure == "SoftDTW":
            for ts1, ts2, ts3 in zip(df1[mask], df2[mask], df3[mask]):
                d1 = soft_dtw(ts1.reshape(-1, 1), c1.reshape(-1, 1), gamma=gamma)
                d2 = soft_dtw(ts2.reshape(-1, 1), c2.reshape(-1, 1), gamma=gamma)
                d3 = soft_dtw(ts3.reshape(-1, 1), c3.reshape(-1, 1), gamma=gamma)
                diff = np.sqrt(d1**2 + d2**2 + d3**2)  # combine like Euclidean norm
                distances.append(diff)

        else:
            raise ValueError("Unsupported distance measure: choose 'Euclidean' or 'SoftDTW'")

    return np.mean(distances)

# Get no-embedding baselines

In [38]:
poor = pd.read_csv("SimData/bank_reserves_outputs_poor.csv", header=None)
middle = pd.read_csv("SimData/bank_reserves_outputs_middle.csv", header=None)
rich = pd.read_csv("SimData/bank_reserves_outputs_rich.csv", header=None)

combined = pd.concat([poor, middle, rich], axis=1)
combined = StandardScaler().fit_transform(combined)

k = 7   
kmeans = KMeans(n_clusters=k, random_state=42)
br_baseline_labels = kmeans.fit_predict(combined)

In [39]:
br_poor_ts = to_time_series_dataset(poor)
br_middle_ts = to_time_series_dataset(middle)
br_rich_ts = to_time_series_dataset(rich)

distance_br_no_embedding_kmeans = ts_cluster_distance(br_baseline_labels, br_poor_ts, br_middle_ts, br_rich_ts, 7, "Euclidean")

In [40]:
ecv_active = pd.read_csv("SimData/epsteinCV_outputs_active.csv", header=None)
ecv_jailed = pd.read_csv("SimData/epsteinCV_outputs_jailed.csv", header=None)
ecv_quiet = pd.read_csv("SimData/epsteinCV_outputs_quiet.csv", header=None)

combined = pd.concat([ecv_active, ecv_jailed, ecv_quiet], axis=1)
combined = StandardScaler().fit_transform(combined)

k = 8   
kmeans = KMeans(n_clusters=k, random_state=42)
ecv_baseline_labels = kmeans.fit_predict(combined)

In [41]:
ecv_active_ts = to_time_series_dataset(ecv_active)
ecv_jailed_ts = to_time_series_dataset(ecv_jailed)
ecv_quiet_ts = to_time_series_dataset(ecv_quiet)

distance_ecv_no_embedding_kmeans = ts_cluster_distance(ecv_baseline_labels, ecv_active_ts, ecv_jailed_ts, ecv_quiet_ts, 8, "Euclidean")

In [42]:
ff_onfire = import_ff_data("SimData/forest_fire_outputs_onfire.csv")
ff_fine = import_ff_data("SimData/forest_fire_outputs_fine.csv")
ff_burned = import_ff_data("SimData/forest_fire_outputs_burned.csv")

combined = pd.concat([ff_onfire, ff_fine, ff_burned], axis=1)
combined = StandardScaler().fit_transform(combined)

k = 4   
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(combined)

In [43]:
ff_onfire_ts = to_time_series_dataset(ff_onfire)
ff_fine_ts = to_time_series_dataset(ff_fine)
ff_burned_ts = to_time_series_dataset(ff_burned)

distance_ff_no_embedding_kmeans = ts_cluster_distance(ecv_baseline_labels, ecv_active_ts, ecv_jailed_ts, ecv_quiet_ts, 8, "Euclidean")

# PCA

In [44]:
label_results = pd.read_csv('bank_reserves_results.csv', index_col=0)

br_poor = pd.read_csv("SimData/bank_reserves_outputs_poor.csv", header=None)
br_poor_ts = to_time_series_dataset(br_poor)

br_middle = pd.read_csv("SimData/bank_reserves_outputs_middle.csv", header=None)
br_middle_ts = to_time_series_dataset(br_middle)

br_rich = pd.read_csv("SimData/bank_reserves_outputs_rich.csv", header=None)
br_rich_ts = to_time_series_dataset(br_rich)

In [45]:
distance_br_pca_kmeans = ts_cluster_distance(label_results.loc["PCA_KMeans"].to_numpy(), br_poor_ts, br_middle_ts, br_rich_ts, 7, "Euclidean")
#distance_br_pca_agglom = ts_cluster_distance(label_results.loc["PCA_Agglom"].to_numpy(), br_poor_ts, br_middle_ts, br_rich_ts, 7, "Euclidean")
#distance_br_pca_spectral = ts_cluster_distance(label_results.loc["PCA_Spectral"].to_numpy(), br_poor_ts, br_middle_ts, br_rich_ts, 7, "Euclidean")

In [46]:
label_results = pd.read_csv('epstein_results.csv', index_col=0)

ecv_active = pd.read_csv("SimData/epsteinCV_outputs_active.csv", header=None)
ecv_active_ts = to_time_series_dataset(ecv_active)

ecv_jailed = pd.read_csv("SimData/epsteinCV_outputs_jailed.csv", header=None)
ecv_jailed_ts = to_time_series_dataset(ecv_jailed)

ecv_quiet = pd.read_csv("SimData/epsteinCV_outputs_quiet.csv", header=None)
ecv_quiet_ts = to_time_series_dataset(ecv_quiet)

In [47]:
distance_ecv_pca_kmeans = ts_cluster_distance(label_results.loc["PCA_KMeans"].to_numpy(), ecv_active_ts, ecv_jailed_ts, ecv_quiet_ts, 8, "Euclidean")
#distance_ecv_pca_agglom = ts_cluster_distance(label_results.loc["PCA_Agglom"].to_numpy(), ecv_active_ts, ecv_jailed_ts, ecv_quiet_ts, 8, "Euclidean")
#distance_ecv_pca_spectral = ts_cluster_distance(label_results.loc["PCA_Spectral"].to_numpy(), ecv_active_ts, ecv_jailed_ts, ecv_quiet_ts, 8, "Euclidean")

In [48]:
label_results = pd.read_csv('forestfire_results.csv', index_col=0)

ff_onfire = import_ff_data("SimData/forest_fire_outputs_onfire.csv")
ff_onfire_ts = to_time_series_dataset(ff_onfire)

ff_fine = import_ff_data("SimData/forest_fire_outputs_fine.csv")
ff_fine_ts = to_time_series_dataset(ff_fine)

ff_burned = import_ff_data("SimData/forest_fire_outputs_burned.csv")
ff_burned_ts = to_time_series_dataset(ff_burned)

In [49]:
distance_ff_pca_kmeans = ts_cluster_distance(label_results.loc["PCA_KMeans"].to_numpy(), ff_onfire_ts, ff_fine_ts, ff_burned_ts, 4, "Euclidean")
#distance_ff_pca_agglom = ts_cluster_distance(label_results.loc["PCA_Agglom"].to_numpy(), ff_onfire_ts, ff_fine_ts, ff_burned_ts, 4, "Euclidean")
#distance_ff_pca_spectral = ts_cluster_distance(label_results.loc["PCA_Spectral"].to_numpy(), ff_onfire_ts, ff_fine_ts, ff_burned_ts, 4, "Euclidean")

# DAE

In [ ]:
label_results = pd.read_csv('bank_reserves_results.csv', index_col=0)
distance_br_dae_kmeans = ts_cluster_distance(label_results.loc["DAE_KMeans"].to_numpy(), br_poor_ts, br_middle_ts, br_rich_ts, 7, "Euclidean")

In [ ]:
label_results = pd.read_csv('epstein_results.csv', index_col=0)
distance_ecv_dae_kmeans = ts_cluster_distance(label_results.loc["DAE_KMeans"].to_numpy(), ecv_active_ts, ecv_jailed_ts, ecv_quiet_ts, 8, "Euclidean")